<a href="https://colab.research.google.com/github/Dkaushani/Automated-Economic-Financial-Dashboard/blob/main/Economic_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Install required libraries
!pip install yfinance plotly

In [ ]:
# Step 2: Import libraries into your script
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
# Step 3 (Fixed): Download historical data and flatten column names
tickers = {
    'CL=F': 'Crude_Oil',
    '^GSPC': 'SP500',
    '^TNX': 'US10Y_Yield'
}

# 1. Download data for all tickers at once
raw_data = yf.download(list(tickers.keys()), start="2020-01-01")['Close']

# 2. Rename columns from ticker symbols (CL=F, ^GSPC, ^TNX) to friendly names
df = raw_data.rename(columns=tickers)

# 3. Inspect columns to confirm they exist
print("Columns in DataFrame:", df.columns.tolist())
df.head()

/tmp/ipykernel_2575/3268975117.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw_data = yf.download(list(tickers.keys()), start="2020-01-01")['Close']
[*********************100%***********************]  3 of 3 completed

Columns in DataFrame: ['Crude_Oil', 'SP500', 'US10Y_Yield']


Ticker,Crude_Oil,SP500,US10Y_Yield
Date,,,
2020-01-02,61.180000,3257.850098,1.882
2020-01-03,63.049999,3234.850098,1.788
2020-01-06,63.270000,3246.280029,1.811
2020-01-07,62.700001,3237.179932,1.827
2020-01-08,59.610001,3253.050049,1.874


In [ ]:
# Step 4: Calculate rolling averages and percentage growth

# 1. Drop any missing values (days when markets were closed)
df = df.dropna()

# 2. Calculate 30-day Rolling Average for Crude Oil
df['Crude_Oil_30MA'] = df['Crude_Oil'].rolling(window=30).mean()

# 3. Calculate year-over-year percentage change for S&P 500
df['SP500_YoY_Change'] = df['SP500'].pct_change(periods=252) * 100

# Inspect the updated dataset structure
df.tail()

Ticker,Crude_Oil,SP500,US10Y_Yield,Crude_Oil_30MA,SP500_YoY_Change
Date,,,,,
2026-07-31,84.669998,7489.720215,4.745,77.307333,17.709226
2026-08-03,80.339996,7600.500000,4.686,77.432000,19.893236
2026-08-04,75.769997,7736.520020,4.627,77.463667,24.022249
2026-08-05,75.220001,7723.549805,4.617,77.530667,22.016162
2026-08-06,78.000000,7709.959961,4.670,77.786000,22.396055


In [ ]:
# Step 5a: Interactive line plot for Crude Oil
fig1 = px.line(df, x=df.index, y=['Crude_Oil', 'Crude_Oil_30MA'],
               labels={'value': 'Price (USD per barrel)', 'Date': 'Date', 'variable': 'Series'},
               title='<b>Crude Oil Spot Price vs. 30-Day Moving Average</b>')

fig1.update_layout(template='plotly_white')
fig1.show()

In [ ]:
# Step 5b: Dual-axis plot comparing S&P 500 and US 10-Year Treasury Yield
fig2 = make_subplots(specs=[[{"secondary_y": True}]])

# Add S&P 500 trace
fig2.add_trace(
    go.Scatter(x=df.index, y=df['SP500'], name="S&P 500 Index", line=dict(color="blue")),
    secondary_y=False,
)

# Add 10-Year Yield trace
fig2.add_trace(
    go.Scatter(x=df.index, y=df['US10Y_Yield'], name="US 10Y Yield (%)", line=dict(color="red", dash="dash")),
    secondary_y=True,
)

# Add titles and labels
fig2.update_layout(title_text="<b>S&P 500 Performance vs. 10-Year US Treasury Yield</b>", template="plotly_white")
fig2.update_yaxes(title_text="<b>S&P 500 Index Value</b>", secondary_y=False)
fig2.update_yaxes(title_text="<b>10-Year Yield (%)</b>", secondary_y=True)

fig2.show()

In [ ]:
# Step 6: Generate summary statistics and save to CSV
summary_stats = df.describe()

# Export dataframe to a CSV file in your Colab workspace
summary_stats.to_csv('economic_summary_stats.csv')

print("Summary statistics saved successfully as 'economic_summary_stats.csv'!")

Summary statistics saved successfully as 'economic_summary_stats.csv'!
